# 04 - Evaluation and Grad-CAM

Evaluate on the **test set only**, once you have picked your final model. Then inspect where the model looks.

**Running on Google Colab?** Upload or clone the project, then run once:

```python
from google.colab import drive
drive.mount("/content/drive")          # optional: keep data and models on Drive
%cd /content/iClaim
%pip install -q -r requirements.txt
```
Use a GPU runtime (Runtime > Change runtime type) for training.

In [ ]:
# --- Setup (works locally and in Google Colab) ---
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

from src.config import load_config, resolve_path, set_seed
cfg = load_config()
set_seed(cfg["project"]["seed"])
print("Project root:", ROOT)

## Test-set evaluation

In [ ]:
from src.evaluate import evaluate_model

metrics = {}
for name, path in [("baseline", "models/baseline.keras"), ("mobilenetv2", "models/mobilenetv2.keras")]:
    if resolve_path(path).exists():
        metrics[name] = evaluate_model(path, name=name, cfg=cfg)

pd.DataFrame(
    [{"model": k, "accuracy": v["accuracy"], "macro_f1": v["macro_f1"], "params": v["parameters"], "ms/img": round(v["ms_per_image"], 1)}
     for k, v in metrics.items()]
)

Confusion matrices and `reports/results.md` are written automatically.

In [ ]:
from IPython.display import Image as IPImage, display
for name in metrics:
    display(IPImage(filename=str(resolve_path(cfg["paths"]["figures_dir"]) / f"confusion_matrix_{name}.png")))

## Grad-CAM: correct and incorrect predictions

In [ ]:
from pathlib import Path
from PIL import Image
from tensorflow import keras
from src.gradcam import make_gradcam_heatmap, overlay_heatmap
from src.predict import class_names_for
from src.utils import preprocess_image

model = keras.models.load_model(resolve_path(cfg["paths"]["best_model"]))
names = class_names_for(cfg)
test_root = resolve_path(cfg["paths"]["processed_data"]) / "test"
files = [(c, p) for c in names for p in sorted((test_root / c).glob("*"))]
rng = np.random.default_rng(cfg["project"]["seed"])
rng.shuffle(files)

rows = []
for true_cls, p in files[:60]:
    batch = preprocess_image(Image.open(p), cfg)
    pred = int(np.argmax(model.predict(batch, verbose=0)[0]))
    rows.append((true_cls, names[pred], p, batch))

correct = [r for r in rows if r[0] == r[1]][:4]
wrong = [r for r in rows if r[0] != r[1]][:4]

def show(items, title):
    if not items:
        print("No examples for:", title); return
    fig, axes = plt.subplots(1, len(items), figsize=(3.4 * len(items), 3.6), squeeze=False)
    for ax, (t, pr, p, batch) in zip(axes[0], items):
        heat = make_gradcam_heatmap(model, batch)
        ax.imshow(overlay_heatmap(Image.open(p).convert("RGB"), heat)); ax.axis("off")
        ax.set_title(f"true: {t}\npred: {pr}", fontsize=9)
    plt.suptitle(title); plt.tight_layout()
    plt.savefig(resolve_path(cfg["paths"]["figures_dir"]) / f"gradcam_{title.split()[0].lower()}.png", dpi=130)
    plt.show()

show(correct, "Correct predictions")
show(wrong, "Wrong predictions")

## Error analysis

- Which class pairs are confused most (see the confusion matrix)?
- Does Grad-CAM focus on the damage or on background/reflections?
- Which image conditions (angle, lighting, distance) cause errors?